# 06 — Analysis & Visualization

Generate all figures and analysis for the report:

1. t-SNE / UMAP embedding plots (random-init vs SSL)
2. Confusion matrices (5×5)
3. Graph attention maps per strategy
4. K-shot learning curve
5. Per-strategy feature importance (RQ1)
6. SSL benefit quantification (RQ2)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.config import *

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Embedding Visualization (t-SNE)

In [ ]:
# TODO: Load embeddings from saved results
# Compare random-init vs SSL-pretrained embeddings

def plot_embeddings(embeddings, labels, title, save_path=None):
    """t-SNE visualization of embeddings colored by strategy class."""
    tsne = TSNE(n_components=2, perplexity=min(30, len(embeddings)-1), random_state=42)
    coords = tsne.fit_transform(embeddings)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, strategy in enumerate(STRATEGY_CLASSES):
        mask = labels == i
        ax.scatter(coords[mask, 0], coords[mask, 1], 
                   label=strategy, alpha=0.7, s=50)
    
    ax.legend(fontsize=11)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Embedding visualization — pending trained models")

## 2. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    """Plot normalized confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=STRATEGY_CLASSES,
                yticklabels=STRATEGY_CLASSES,
                ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Confusion matrix — pending evaluation results")

## 3. K-Shot Learning Curve

In [ ]:
def plot_kshot_curve(k_values, f1_means, f1_stds, save_path=None):
    """Plot F1 as a function of K."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(k_values, f1_means, yerr=f1_stds, 
                marker='o', capsize=5, linewidth=2)
    ax.set_xlabel('K (shots per class)', fontsize=12)
    ax.set_ylabel('Macro-F1', fontsize=12)
    ax.set_title('Few-Shot Performance vs. Support Set Size')
    ax.set_xticks(k_values)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("K-shot curve — pending ablation results")

## 4. Feature Importance (RQ1)

In [ ]:
def plot_feature_importance(spatial_f1, temporal_f1, full_f1, save_path=None):
    """Bar chart comparing spatial vs temporal F1 per strategy."""
    x = np.arange(len(STRATEGY_CLASSES))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width, spatial_f1, width, label='Spatial Only', color='steelblue')
    ax.bar(x, temporal_f1, width, label='Temporal Only', color='coral')
    ax.bar(x + width, full_f1, width, label='Full (S+T)', color='forestgreen')
    
    ax.set_xlabel('Strategy')
    ax.set_ylabel('F1 Score')
    ax.set_title('Per-Strategy F1 by Feature Type (RQ1)')
    ax.set_xticks(x)
    ax.set_xticklabels(STRATEGY_CLASSES, rotation=15)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Feature importance plot — pending RQ1 ablation results")

## 5. SSL Benefit (RQ2)

In [ ]:
def plot_ssl_comparison(results_dict, save_path=None):
    """Compare random init vs SSL vs SSL+aux."""
    models = list(results_dict.keys())
    f1_scores = [results_dict[m]['macro_f1'] for m in models]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(models, f1_scores, color=['gray', 'steelblue', 'forestgreen'])
    
    for bar, score in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.1%}', ha='center', fontsize=12)
    
    ax.set_ylabel('Macro-F1')
    ax.set_title('Pre-Training Ablation (RQ2)')
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("SSL comparison — pending RQ2 ablation results")